# 06 · Validação multianual de Iatan 2

Aplicamos o desenho de ablação do Exp33 a **2023, 2024 e 2025** com treino expansivo desde 2011. A referência usa uma origem por ano. O teste de sensibilidade trimestral reajusta o modelo em janeiro, abril, julho e outubro, sempre com dados anteriores à origem. As duas cadências usam as mesmas horas elegíveis. O alvo é o CO₂ da própria hora, em toneladas métricas por hora, com a geração dessa hora disponível.

- **A** usa a geração da hora.
- **B** acrescenta o histórico de carga no ciclo.
- **C** acrescenta as probabilidades e os atributos do modelo oculto de Markov (HMM).
- **D** acrescenta os atributos das transições do grafo de sucessão direta (DFG).

O XGBoost mantém os mesmos parâmetros e os quatro cenários usam as mesmas horas elegíveis em cada ano. O escalonador, o HMM, o DFG e a imputação dos atributos são ajustados no treino de cada janela. A configuração e as fontes estão no [protocolo da Fase 5](../../configs/phase5_iatan.json).

Na ablação trimestral de lags e processo, usamos A como referência e avaliamos mais seis combinações. Os lags são as oito defasagens de 2, 20, 23, 24, 26, 29, 41 e 52 horas selecionadas no [Notebook 04](04_estudo_temporal.ipynb), calculadas no calendário horário antes de filtrar as horas ON. A lista da Fase 3 foi selecionada com dados de 2025, o que torna esta comparação retrospectiva. As combinações L, LH e LD usam somente essas defasagens como histórico explícito.

| Cenário | Entradas do XGBoost |
|---|---|
| A | Geração da própria hora |
| L | Geração + lags |
| LH | Geração + lags + HMM |
| LD | Geração + lags + DFG |
| H | Geração + HMM + DFG |
| F | Geração + DFG |
| E | Geração + HMM |

A ablação recupera os mesmos modelos operacionais e as previsões de A da rodada trimestral A–D. Valida as horas e os alvos, reproduz A, C e D para conferir os artefatos de origem e ajusta as seis combinações novas apenas no treino de cada janela. O DFG deriva dos estados do HMM. F e LD entregam ao XGBoost os atributos do grafo, sem as colunas diretas do HMM.

A comparação de comprimento do treino mantém o cenário A, as 12 origens trimestrais e as mesmas horas de teste. Em cada origem, reaplicamos o modelo expansivo arquivado e ajustamos o mesmo pipeline de imputação e XGBoost com as horas elegíveis nos três anos de calendário imediatamente anteriores. O teste é retrospectivo.

A ablação operacional mantém **A** como referência e ajusta **R** (geração, rampa e tempo desde o religamento), **RH** (R com os atributos do HMM) e **RD** (R com os atributos do DFG). A rampa é a carga da hora atual menos a anterior quando ambas são ON, válidas e consecutivas. A idade começa em zero na primeira hora ON depois de uma hora OFF observada. Calculamos os dois atributos no calendário completo e mantemos ausentes os valores sem histórico suficiente. Em cada trimestre, RH e RD usam a representação operacional já ajustada no treino arquivado. O DFG deriva dos estados do HMM e fornece apenas os atributos do grafo ao XGBoost em RD.

## Insumos e execução

1. **Prepare o ambiente.** Instale as [dependências do projeto](../../requirements-iatan.txt) e coloque os 15 Parquets anuais de `iatan_unit2_2011.parquet` a `iatan_unit2_2025.parquet` em `data/interim/03A04_multiyear/`, na raiz do projeto. A execução confere os arquivos pelo [manifesto](../../configs/iatan_inputs.json). A rodada de referência usou Windows com Python 3.12. Trate a rodada no Colab com Linux e Python 3.13 como comparação entre ambientes.

2. **Execute a modelagem.** Inicie o kernel na raiz do projeto ou defina `IATAN_PROJECT_ROOT` com o caminho dessa raiz. No Colab, monte o Drive se os arquivos estiverem nele e aponte a variável para a pasta que contém `configs/` e `iatan/`. Escolha `annual` para a referência, `quarterly` para a análise trimestral A–D, `ablation` para comparar lags, HMM e DFG, `training_window` para comparar os dois comprimentos de treino de A ou `operational_ablation` para comparar A, R, RH e RD. Para os três modos de comparação, indique a pasta da rodada trimestral A–D concluída, que contém `resultados/`, em `IATAN_PHASE5_QUARTERLY_REFERENCE`. Use as versões numéricas da rodada de origem. Cada rodada grava resultados e uma base MLflow própria na subpasta `outputs/` correspondente ao modo. A ablação operacional usa `outputs/phase_05_operational_ablation/`.

3. **Consulte as tabelas.** Execute a célula de métricas depois da conclusão da modelagem na mesma sessão. Para ler uma rodada já concluída após reiniciar, defina `resultados` como um `Path` para a subpasta `resultados/` dessa rodada.

In [ ]:
import os
from pathlib import Path
import sys

root = Path(os.environ.get('IATAN_PROJECT_ROOT', str(Path.cwd()))).expanduser().resolve()
if not ((root / 'configs/phase5_iatan.json').is_file() and (root / 'iatan/phase5.py').is_file()):
    raise FileNotFoundError(
        f'Raiz do projeto não encontrada em {root}. '
        'Inicie o kernel na raiz do repositório ou defina IATAN_PROJECT_ROOT com esse caminho.'
    )
sys.path.insert(0, str(root))

from iatan import phase5

modo = 'annual'  # Opções: annual, quarterly, ablation, training_window, operational_ablation.
if modo in {'ablation', 'training_window', 'operational_ablation'}:
    from iatan import phase5_ablation, phase5_operational_ablation, phase5_training_window

    referencia = os.environ.get('IATAN_PHASE5_QUARTERLY_REFERENCE')
    if not referencia:
        raise ValueError('Defina IATAN_PHASE5_QUARTERLY_REFERENCE com a pasta da rodada trimestral A–D.')
    comparacao = {
        'ablation': phase5_ablation,
        'training_window': phase5_training_window,
        'operational_ablation': phase5_operational_ablation,
    }[modo]
    resultados = comparacao.run(root, referencia) / 'resultados'
else:
    resultados = phase5.run(root, cadence=modo) / 'resultados'

## Métricas e contrastes

As métricas são RMSE, MAE e viés em t CO₂/h. O contraste usa o RMSE do cenário candidato menos o da referência. Valores negativos favorecem o candidato. Na rodada A–D, **D − C** mede a contribuição incremental do DFG na presença de histórico e HMM, e **D − A** compara o conjunto completo de atributos com a geração da própria hora. Na ablação, **E − A** e **F − A** comparam o HMM e o DFG com A separadamente. Na ablação operacional, **R − A** mede a contribuição conjunta da rampa e do religamento, enquanto **RH − R** e **RD − R** medem os incrementos de HMM e DFG sobre R.

As rodadas trimestrais apresentam as métricas por trimestre, por ano e no conjunto dos três anos. Os contrastes usam IC de 95% por bootstrap pareado em blocos de 7 dias de calendário e sensibilidades de 1 e 3 dias. A ablação e a comparação de comprimento apresentam os contrastes também por trimestre. Nesta última, o contraste é o RMSE do treino de três anos menos o do expansivo. A reamostragem usa as previsões dos modelos já ajustados. As horas têm dependência temporal e os treinos se sobrepõem. Trata-se de uma avaliação retrospectiva em Iatan 2.

In [ ]:
if 'resultados' not in globals():
    raise RuntimeError(
        'Execute a célula de modelagem primeiro ou defina resultados '
        'como um Path para a subpasta resultados/ de uma rodada concluída.'
    )

from IPython.display import display
import pandas as pd

metricas = pd.read_csv(resultados / 'metricas.csv')
contrastes = pd.read_csv(resultados / 'contrastes.csv')

periodo = ['nivel', 'janela'] if 'janela' in metricas.columns else ['ano']
display(metricas[periodo + ['cenario', 'n', 'rmse', 'mae', 'vies']].round(3))
colunas_ic = ['periodo', 'contraste', 'n_horas_on', 'delta_rmse', 'ic95_inferior', 'ic95_superior']
display(contrastes.loc[contrastes['bloco_dias'].eq(7), colunas_ic].round(3))

## Referências

- Desenho e proveniência. IND2622, Aula 08, pp. 23–32 e 52–56. [Notebook 05, Exp33](05_modelagem_co2_mlflow.ipynb).
- Avaliação temporal. [Cerqueira, Torgo e Mozetič (2020)](https://doi.org/10.1007/s10994-020-05910-7). Origem móvel adaptada à estimação da hora corrente. [Hyndman e Athanasopoulos, seção 5.10](https://otexts.com/fpp3/toolbox.html#time-series-cross-validation).
- Dependência temporal e reamostragem em blocos. [Künsch (1989)](https://doi.org/10.1214/aos/1176347265).
- Estimação e estados operacionais. [Chen e Guestrin (2016)](https://arxiv.org/abs/1603.02754) e [Rabiner (1989)](https://doi.org/10.1109/5.18626).
- Registros de eventos e descoberta de processos. IND2622, Aula 07, pp. 62–72, e [Diba et al. (2020)](https://doi.org/10.1002/widm.1346).
- Exemplo de mineração de processos com dados de sensores. [Moradbeikie et al. (2025)](https://doi.org/10.3390/systems13110935).
- Dados horários. [U.S. EPA, Power Sector Emission Data Guide](https://www.epa.gov/power-sector/power-sector-emission-data-guide).
- Rampa entre horas consecutivas. [Gu, Sward e Zhang (2023), Eq. 2](https://doi.org/10.1039/D3VA00191A).